# 기후·토지이용 변화와 화분매개곤충: 서식지 이동, 꽃과의 공존 관계, 군집 구성 변화 예측

**강의안 `SDM_HandsOn_Slides.pdf` (`Slides 1–43`) 대응**

- **분석 자료**: 제3~5차 전국자연환경조사(NES) 곤충·식물 `Presence-Only (PO)` 기록 및 남한 내륙 1 km 격자(`84,576`개, `EPSG:5186`) 환경변수 22종(`Slides 1–3`)
- **적합 원리**: 출현 지점과 무작위 배경 지점의 환경을 비교하는 `Presence–Background (PB)` 모형(MaxEnt 계열) 적용

### 실습 구성
1. **`[실습]` 단일 사례** — 산지성 `FG03 뒤영벌속` 중심 단계별 직접 계산(`pandas`·`numpy`·`rasterio`·`matplotlib`)
2. **`[전체 비교]` 다분류군 합성** — `sdm_core` 함수로 **5개 목**, **12개 기능군(`FG01`–`FG12`)**, **8개 상호작용 쌍(및 274종 식물 × 139종 곤충 군집)**, **4가지 군집 행렬** 비교

---

1. **Step 1 — 분포 특성과 서식 환경 (`Slides 4–14`)**: 기능군별 출현 격자·분포 고도·기온 한계(`Bio5`/`Bio6`) 확인 및 2020년 환경 구배 지도화
2. **Step 2 — 서식 적합도 모형과 2050년 분포 변화 (`Slides 15–26`)**: MaxEnt `L/Q/H` 반응 추정, 50 km 공간 `block` 교차검증, 기온 한계 반영 2050년 서식지 투영
3. **Step 3 — 식물–화분매개자 공존 서식지 변화 (`Slides 27–31`)**: 서식 적합지 중첩도(Schoener's $D$), 공존 변화 4유형 분류, 전국 공존 피난처 비교
4. **Step 4 — 군집 종조성 변화 (`Slides 32–41`)**: 적층 종분포모형(`SSDM`)과 일반화 비유사도 모형(`GDM`) 결합 및 Baselga 종 교체·종 감소 분할

## 실행 환경 (`Slide 3`)

[![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/alanbseo/SDMworkshop/main?urlpath=lab/tree/SDM_HandsOn_Workshop_KO.ipynb)

| 구분 | `MyBinder` (웹 실행) | `Local` (개인 PC) |
| :--- | :--- | :--- |
| **실행** | 상단 `Launch Binder` 배지 클릭 (`Python 3.11`·`data/` 사전 구성, 기동 약 `20~40초`) | 저장소 복제 후 가상환경에서 `JupyterLab` 실행 |
| **보관** | 약 `10분` 무입력 시 `session` 종료 → 종료 전 `.ipynb` 우클릭 **`Download`** 필요 | 로컬 저장장치에 자동 유지 |

```bash
# Local 실행 시
git clone https://github.com/alanbseo/SDMworkshop.git
cd SDMworkshop
python3 -m venv .venv && source .venv/bin/activate   # Windows: .venv\Scripts\activate
pip install -r requirements.txt
jupyter lab SDM_HandsOn_Workshop_KO.ipynb
```
*(`Linux` local 환경은 `fonts-nanum` 별도 설치)*

## 0. 환경 설정
- 작업 경로·한글 글꼴 설정 및 `scripts/sdm_core.py` 핵심 함수 적재

In [ ]:
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.colors import ListedColormap, BoundaryNorm

warnings.filterwarnings("ignore", message=".*encountered in matmul", category=RuntimeWarning)  # 일부 BLAS 환경의 행렬곱 수치 경고 숨김

WORKSHOP_ROOT = os.path.abspath(".") if os.path.isdir("scripts") else os.path.abspath("..")
sys.path.insert(0, os.path.join(WORKSHOP_ROOT, "scripts"))

from sdm_core import (
    DATA_DIR, DEFAULT_DRIVERS, GFO_COLORS,
    load_pa_tables, load_env_scenario,
    build_maxent_features, fit_maxent_logistic, predict_suitability, evaluate_roc_tss,
    train_and_validate_sdm, compute_response_curve,
    project_to_scenario, analyze_bipartite_overlap,
    eval_ispline_3, fit_gdm_ispline,
    vec_to_2d, plot_4state_map,
)

# 22개 환경 예측변수 국문 label 사전
DRIVER_LABELS_KO = {
    "bioclim1": "연평균기온 (bioclim1, °C)",
    "bioclim2": "평균 일교차 (bioclim2, °C)",
    "bioclim3": "등온성 (bioclim3, %)",
    "bioclim13": "최습월 강수량 (bioclim13, mm)",
    "bioclim14": "최건월 강수량 (bioclim14, mm)",
    "solarradiation": "일사량 (solarradiation, MJ/m²/d)",
    "relative_humidity": "상대습도 (relative_humidity, %)",
    "VPD": "수증기압차 (VPD, hPa)",
    "AI": "건조지수 (AI)",
    "WB_spr": "봄철 수분수지 (WB_spr, mm)",
    "CWD": "기후적 수분부족량 (CWD, mm)",
    "elevation": "해발고도 (elevation, m)",
    "slope": "경사도 (slope, °)",
    "soildeep3": "유효토심 (soildeep3)",
    "soildra1": "토양배수등급 (soildra1)",
    "soilstone": "토양자갈함량 (soilstone)",
    "frac_0": "시가화 비율 (frac_0, 100m)",
    "frac_1": "농경지 비율 (frac_1, 100m)",
    "frac_2": "산림 비율 (frac_2, 100m)",
    "frac_3": "초지 비율 (frac_3, 100m)",
    "edge_FO": "산림-개방지 경계밀도 (edge_FO, km/km²)",
    "shannon": "토지피복 다양성 (shannon, H')",
}

nanum_ttf = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if os.path.exists(nanum_ttf):  # Linux / MyBinder 환경 나눔고딕 직접 등록
    fm.fontManager.addfont(nanum_ttf)
plt.rcParams["font.sans-serif"] = ["Apple SD Gothic Neo", "AppleGothic", "Malgun Gothic", "NanumGothic", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

is_binder = ("BINDER_SERVICE_HOST" in os.environ) or (os.environ.get("USER") == "jovyan")
run_env_label = "MyBinder" if is_binder else "local"
avail_fonts = {f.name for f in fm.fontManager.ttflist}
active_font = next((fn for fn in plt.rcParams["font.sans-serif"] if fn in avail_fonts), "DejaVu Sans")
print(f"실행 환경: {run_env_label} | Python {sys.version.split()[0]} | 작업 경로: {WORKSHOP_ROOT} | 글꼴: {active_font}")
%matplotlib inline

## Step 1-A. 출현 기록으로 본 분포 고도와 기온 범위 (`Slides 4–9`)
- 산지성 **`FG03` 뒤영벌속(*Bombus*)**의 출현 격자 수, 평균 해발고도, 분포지 관찰 여름 최고기온(`Bio5`) 상한·겨울 최저기온(`Bio6`) 하한 확인

In [ ]:
# [실습 1-A] FG03 뒤영벌속 출현 격자·분포 고도·기온 한계 확인
pa_subgroups = pd.read_csv(os.path.join(DATA_DIR, "PA_Pollinator_Subgroups_1km.csv"))
bg_grid = pd.read_csv(os.path.join(DATA_DIR, "Background_Grid_1km.csv"))  # 내륙 84,576개 1 km 격자 좌표·고도
with open(os.path.join(DATA_DIR, "Thermal_Limits.json"), encoding="utf-8") as f:
    thermal_limits = json.load(f)  # 분류군별 관찰 기온 상·하한(Bio5 / Bio6)

print("PA_Pollinator_Subgroups_1km.csv 열:", list(pa_subgroups.columns))
print(f"남한 내륙 1 km 총 격자 수: {len(bg_grid):,}개 | 고도 결측치: {int(bg_grid['elevation_m'].isna().sum())}개")

bombus_df = pa_subgroups[pa_subgroups["subgroup_code"] == "FG03"]
bombus_ids = bombus_df["grid_id"].to_numpy(dtype=np.int32)  # grid_id(0~84,575) = bg_grid 행 번호
bombus_tl = thermal_limits["subgroups"]["뒤영벌속"]

mean_elev_bombus = bg_grid.loc[bombus_ids, "elevation_m"].mean()
mean_elev_korea = bg_grid["elevation_m"].mean()

print(f"FG03 뒤영벌속 출현 격자(NES 3~5차): {len(bombus_df):,}개 (5차 조사: {int(bombus_df['has_5th'].sum()):,}개)")
print(f"평균 해발고도 — FG03 뒤영벌속: {mean_elev_bombus:.1f} m vs. 내륙 전체: {mean_elev_korea:.1f} m")
print(f"관찰 기온 한계 — 하계 상한(Bio5) T_max: {bombus_tl['hot_limit_C']} °C | 동계 하한(Bio6) T_min: {bombus_tl['cold_limit_C']} °C")
bombus_df.head(5)

- `FG03 뒤영벌속(*Bombus*)`: 내륙 `927`개 격자 출현(5차 `380`개), 평균 해발고도 `309.3 m`(내륙 평균 `285.9 m` 상회), 하계 상한(`Bio5` $T_{\max}$) `33.41 °C`의 저온성 지위 확인

In [ ]:
# [전체 비교 1-A] 12개 기능군(FG01~FG12) 출현 격자 수·분포 고도·기온 한계 비교
pa_orders, pa_subgroups, pa_plants, bg_grid, thermal_limits = load_pa_tables()

SUBGROUP_LIST = [
    ("FG03", "뒤영벌속", "Bombus", "Hymenoptera"),
    ("FG04", "꿀벌속", "Apis", "Hymenoptera"),
    ("FG05", "기타 야생꿀벌과", "Other Apidae", "Hymenoptera"),
    ("FG06", "단독성 야생벌류", "Solitary Bees", "Hymenoptera"),
    ("FG01", "꽃등에과", "Syrphidae", "Diptera"),
    ("FG02", "재니등에과", "Bombyliidae", "Diptera"),
    ("FG07", "온대 노린재류", "Temperate Hemiptera", "Hemiptera"),
    ("FG08", "큰별노린재과", "Southern Largidae", "Hemiptera"),
    ("FG09", "호랑나비과", "Papilionidae", "Lepidoptera"),
    ("FG10", "네발나비과", "Nymphalidae", "Lepidoptera"),
    ("FG11", "기타 나비·나방류", "Other Lepidoptera", "Lepidoptera"),
    ("FG12", "하늘소과", "Cerambycidae", "Coleoptera"),
]

rows_1a = []
for scode, skr, sen, oen in SUBGROUP_LIST:
    sub = pa_subgroups[pa_subgroups["subgroup_kr"] == skr]
    tl = thermal_limits["subgroups"][skr]
    rows_1a.append({
        "기능군_code": scode,
        "상위_목": oen,
        "기능군_국문명": skr,
        "기능군_영문명": sen,
        "출현격자_3_5차": len(sub),
        "출현격자_5차": int(sub["has_5th"].sum()),
        "평균고도_m": round(float(bg_grid.loc[sub["grid_id"], "elevation_m"].mean()), 1),
        "하계상한_Bio5_C": tl["hot_limit_C"],
        "동계하한_Bio6_C": tl["cold_limit_C"],
    })

pd.DataFrame(rows_1a)

- **벌목(`Hymenoptera`) 내 대비**: 산지 저온성 `FG03 뒤영벌속`(`Bio5` 상한 `33.41 °C`, 평균 고도 `309.3 m`) vs. 광온성 `FG04 꿀벌속`(`41.53 °C`, `272.0 m`)
- **노린재목(`Hemiptera`) 내 대비**: 내륙 광역 `FG07 온대 노린재류`(`2,025`개 격자, 상한 `33.41 °C`, 동계 하한 `-23.33 °C`) vs. 남해안 아열대성 `FG08 큰별노린재과`(`124`개 격자, 상한 `35.86 °C`, 동계 하한 `-13.96 °C`)

## Step 1-B. 기후·지형·경관 환경 구배 (`Slides 12–14`)
- **공통 예측변수(22개 `raster`)**: 기후·복사·수분수지(11종), 지형·토양(5종), 100 m 토지피복·경관구조(6종; `SDM`과 `GDM` 공유)
- **모형별 보조 입력**: `bioclim5`·`bioclim6`(기온 한계 제약) 및 `frac_6`(수역 비율 $\ge 0.5$ 제외)는 **SDM 사후 제약**, 지점 간 거리 `Geo_Distance`는 **GDM 공간 변수**(`SDM`에서는 50 km 교차검증 분할에만 사용)

In [ ]:
# [실습 1-B] 2020 연평균기온(bioclim1.tif) 내륙 격자 추출 및 FG03 출현 지점 중첩
with rasterio.open(os.path.join(DATA_DIR, "land_mask_1km.tif")) as src_mask:
    mask_raw = src_mask.read(1)
    mask_2d = (~np.isnan(mask_raw)) & (mask_raw > 0)  # 내륙 유효 화소(84,576개) boolean mask

with rasterio.open(os.path.join(DATA_DIR, "env_2020", "bioclim1.tif")) as src_b1:
    bio1_2d_raw = src_b1.read(1).astype(np.float32)

bio1_1d = bio1_2d_raw[mask_2d]  # 2차원 지도 -> 1차원 내륙 vector 변환
print(f"2D raster 크기: {bio1_2d_raw.shape} | 내륙 유효 격자: {mask_2d.sum():,}개 | 결측치(NaN): {int(np.isnan(bio1_1d).sum())}개")
print(f"연평균기온 — FG03 출현 격자: {bio1_1d[bombus_ids].mean():.2f} °C vs. 내륙 전체: {bio1_1d.mean():.2f} °C")

bio1_map = np.full(mask_2d.shape, np.nan, dtype=np.float32)
bio1_map[mask_2d] = bio1_1d  # 1차원 내륙 vector -> 2차원 지도 복원

fig, ax = plt.subplots(figsize=(4.8, 5.2))
im = ax.imshow(bio1_map, cmap="RdYlBu_r", vmin=np.nanpercentile(bio1_map, 2), vmax=np.nanpercentile(bio1_map, 98))
ax.scatter(bombus_df["col"], bombus_df["row"], s=3.5, c="black", alpha=0.5, label="FG03 뒤영벌속 출현(PO)")
ax.set_title("2020 연평균기온(bioclim1) 및 FG03 출현 격자", fontsize=10.5, fontweight="bold")
ax.axis("off")
ax.legend(loc="lower right", fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.03, label="°C")
plt.tight_layout()
plt.show()

- 전체 1 km GeoTIFF `raster`: `(602, 342)` 격자 중 내륙 유효 화소 `84,576`개 일치(결측치 `0`개)
- `FG03 뒤영벌속` 출현 격자 연평균기온(`12.00 °C`): 내륙 평균(`12.28 °C`)보다 서늘한 백두대간·산지 축 중심 분포

In [ ]:
# [전체 비교 1-B] 2020년 22개 환경 예측변수 적재 및 핵심 5개 환경 구배 비교
env_2020 = load_env_scenario("env_2020")  # X_raw(84,576 x 22) 및 Bio5·Bio6·수역 비율 일괄 적재
print("X_raw 행렬 크기 (내륙 격자 수, 예측변수 수):", env_2020["X_raw"].shape)

show_vars = [
    ("bioclim1", "RdYlBu_r", "연평균기온 (bioclim1, °C)"),
    ("WB_spr", "BrBG", "봄철 수분수지 (WB_spr, mm)"),
    ("elevation", "terrain", "해발고도 (elevation, m)"),
    ("frac_2", "Greens", "산림 비율 (frac_2, 100m)"),
    ("edge_FO", "PuBuGn", "산림-개방지 경계밀도 (edge_FO, km/km²)"),
]

fig, axes = plt.subplots(1, 5, figsize=(15.5, 3.6))
for ax, (vcode, cm, ktitle) in zip(axes, show_vars):
    arr = vec_to_2d(env_2020["layers"][vcode], mask_2d)
    im = ax.imshow(arr, cmap=cm, vmin=np.nanpercentile(arr, 2), vmax=np.nanpercentile(arr, 98))
    ax.set_title(ktitle, fontsize=9.2, fontweight="bold")
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
plt.tight_layout()
plt.show()

- 22개 예측변수 행렬 `(84576, 22)` 적재 확인: 거시 기후(`bioclim1`, `WB_spr`)·지형(`elevation`) 구배와 100 m 토지피복 기반 산림 비율(`frac_2`)·임연부 경계밀도(`edge_FO`)의 국지 이질성 포착

## Step 2-A. 서식 적합도 모형: 환경 반응 추정과 미조사 지역 예측력 검증 (`Slides 15–18`)
`train_and_validate_sdm` 내부 절차를 **`FG03 뒤영벌속`**에 단계별 적용:
1. **출현–배경 구성**: 출현 격자(`y = 1`) vs. 무작위 배경 격자 `10,000`개(`y = 0`, 전국 가용 환경 표본)
2. **50 km 공간 `block` 분할**: `EPSG:5186` 좌표 기준 권역 분할 후 1/3(**`Fold 0`**)을 독립 검증용으로 분리(공간 자기상관 과대평가 방지; Roberts et al., 2017)
3. **`L/Q/H` 기저 확장(`88`개)**: 직선형(L, `22`)·최적 범위형(Q, `22`)·문턱형(`hinge`, H, `44`) 변환 — 표준화 기준(`scaler`)은 **훈련 권역(`Fold 1 & 2`)에서만** 산출해 검증 누수 차단
4. **정규화 적합 및 평가**: 출현·배경 동일 총가중치 L1/L2 정규화 `logistic` 회귀(`beta_reg = 1.5`) 적합 후 **AUC** 및 **Max-TSS 임계치 $\tau$**(Allouche et al., 2006) 산출

In [ ]:
# [실습 2-A1] FG03 뒤영벌속 SDM 적합 및 독립 50 km 공간 block(Fold 0) 검증
rng = np.random.default_rng(42)
pres_ids = np.unique(bombus_ids)
bg_ids = rng.choice(bg_grid["grid_id"].to_numpy(dtype=np.int32), size=10000, replace=False)

# 1) 50 km x 50 km 공간 block 생성 및 3-fold 배정
xs, ys = bg_grid["x_5186"].to_numpy(), bg_grid["y_5186"].to_numpy()
block_code = np.floor((xs - xs.min()) / 50000.0).astype(int) * 100 + np.floor((ys - ys.min()) / 50000.0).astype(int)
uniq_blocks = np.unique(block_code)
rng.shuffle(uniq_blocks)
block_to_fold = {b: (i % 3) for i, b in enumerate(uniq_blocks)}
cell_fold = np.array([block_to_fold[b] for b in block_code], dtype=int)

# 2) 훈련 권역(Fold 1·2)만으로 표준화 기준(scaler) 산출 후 검증 권역(Fold 0) 변환
X_raw_2020 = env_2020["X_raw"]
_, scaler_train = build_maxent_features(X_raw_2020[cell_fold != 0], scaler=None, use_hinge=True)

tr_pres, te_pres = pres_ids[cell_fold[pres_ids] != 0], pres_ids[cell_fold[pres_ids] == 0]
tr_bg, te_bg = bg_ids[cell_fold[bg_ids] != 0], bg_ids[cell_fold[bg_ids] == 0]

X_tr_pres, _ = build_maxent_features(X_raw_2020[tr_pres], scaler=scaler_train)
X_tr_bg, _ = build_maxent_features(X_raw_2020[tr_bg], scaler=scaler_train)
X_te_pres, _ = build_maxent_features(X_raw_2020[te_pres], scaler=scaler_train)
X_te_bg, _ = build_maxent_features(X_raw_2020[te_bg], scaler=scaler_train)
print(f"기저 특징 확장: 원시 변수 {X_raw_2020.shape[1]}개 -> L/Q/H 특징 {X_tr_pres.shape[1]}개")
print(f"Fold 0 분할: 훈련 출현={len(tr_pres)}, 검증 출현={len(te_pres)} | 훈련 배경={len(tr_bg)}, 검증 배경={len(te_bg)}")

# 3) 정규화 logistic 회귀 적합 및 독립 검증 권역 평가
params_fold0 = fit_maxent_logistic(X_tr_pres, X_tr_bg, beta_reg=1.5)
prob_te_pres = predict_suitability(params_fold0, X_te_pres, env_2020["water_frac"][te_pres])
prob_te_bg = predict_suitability(params_fold0, X_te_bg, env_2020["water_frac"][te_bg])

fold0_metrics = evaluate_roc_tss(prob_te_pres, prob_te_bg)
print(f"[FG03 독립 검증 Fold 0] AUC = {fold0_metrics['auc']:.3f}, Max-TSS = {fold0_metrics['tss']:.3f}, 임계치 tau = {fold0_metrics['threshold']:.3f}")

- 22개 원시 환경변수가 `88`개 `L/Q/H` 특징으로 확장되며, 독립 검증 권역(`Fold 0`)에서 `AUC = 0.618`, `Max-TSS = 0.191`, `tau = 0.451` 확인(3-겹 통합 공간 CV `AUC = 0.628`, `Slide 26`)

In [ ]:
# [실습 2-A2] 나머지 21개 변수를 중앙값으로 고정한 FG03 연평균기온(bioclim1) 반응곡선
bombus_model = train_and_validate_sdm(bombus_ids, env_2020, bg_grid, beta_reg=1.5, use_hinge=True)  # 3-겹 CV 및 최종 모형 적합(Step 2-B 재사용)
params_bombus = bombus_model["params"]
scaler_full = bombus_model["scaler"]
tau_cv = bombus_model["threshold"]  # 3-겹 통합 교차검증 Max-TSS 임계치

var_idx = DEFAULT_DRIVERS.index("bioclim1")
X_profile = np.tile(np.median(X_raw_2020, axis=0), (100, 1))  # 22개 변수 전국 중앙값 행렬(100행)
temp_axis = np.linspace(np.percentile(X_raw_2020[:, var_idx], 1), np.percentile(X_raw_2020[:, var_idx], 99), 100)
X_profile[:, var_idx] = temp_axis  # bioclim1 열만 1%~99% 분위수 구간으로 변경

X_profile_feat, _ = build_maxent_features(X_profile, scaler=scaler_full)
suit_curve = predict_suitability(params_bombus, X_profile_feat)

fig, ax = plt.subplots(figsize=(5.6, 3.3))
ax.plot(temp_axis, suit_curve, lw=2.4, color=GFO_COLORS["slate"], label="FG03 뒤영벌속 상대 적합도")
ax.axhline(tau_cv, color=GFO_COLORS["terracotta"], ls="--", label=f"교차검증 임계치 tau = {tau_cv:.2f}")
ax.axvline(temp_axis[suit_curve.argmax()], color="gray", ls=":", lw=1)
print(f"최적 연평균기온: {temp_axis[suit_curve.argmax()]:.1f} °C (적합도 {suit_curve.max():.2f}) | 양 끝: {suit_curve[0]:.2f} ({temp_axis[0]:.1f} °C), {suit_curve[-1]:.2f} ({temp_axis[-1]:.1f} °C) | 임계치 tau = {tau_cv:.3f}")
ax.set_xlabel("연평균기온 (bioclim1, °C)")
ax.set_ylabel("상대 서식 적합도 (Suitability)")
ax.set_title("연평균기온에 대한 서식 적합도 반응 (FG03 뒤영벌속)", fontweight="bold")
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()

- `FG03 뒤영벌속` 기온 반응곡선: 약 `10.4 °C`에서 최대(`0.64`)인 단봉형 반응 후 고온 저지대 방향으로 감소해 약 `14.5 °C` 이상에서 교차검증 임계치(`tau = 0.485`) 하회
- **상대 변수 중요도(`var_imp`)**: 표준화 1 SD 기준 `L/Q/H` 계수 절댓값 합($S_k = |\hat{\beta}_{k,\text{L}}| + |\hat{\beta}_{k,\text{Q}}| + |\hat{\beta}_{k,\text{H1}}| + |\hat{\beta}_{k,\text{H2}}|$) 비중(합계 `100%`; `TARGET_ORDER_EN` 변경 시 다른 목 확인 가능)

In [ ]:
# [전체 비교 2-A] 선택한 곤충 목의 3-겹 공간 CV 성능 및 상위 3개 환경변수 반응곡선
ORDER_EN_TO_KR = {
    "Hymenoptera": "벌목",
    "Diptera": "파리목",
    "Lepidoptera": "나비목",
    "Hemiptera": "노린재목",
    "Coleoptera": "딱정벌레목",
}
TARGET_ORDER_EN = "Hymenoptera"  # Diptera, Lepidoptera, Hemiptera, Coleoptera로 변경 가능
TARGET_ORDER_KR = ORDER_EN_TO_KR[TARGET_ORDER_EN]

order_pres_ids = pa_orders.loc[pa_orders["order_kr"] == TARGET_ORDER_KR, "grid_id"].to_numpy()
order_model = train_and_validate_sdm(order_pres_ids, env_2020, bg_grid, beta_reg=1.5, use_hinge=True)
cv = order_model["cv_metrics"]
print(f"[{TARGET_ORDER_EN} ({TARGET_ORDER_KR})] 3-겹 50 km 공간 block CV: AUC = {cv['auc']:.3f}, TSS = {cv['tss']:.3f}, 임계치 = {cv['threshold']:.3f}")

top_vars = sorted(order_model["var_imp"].items(), key=lambda x: -x[1])

fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
axes[0].plot(cv["fpr_curve"], cv["tpr_curve"], lw=2, color=GFO_COLORS["ochre"], label=f"AUC={cv['auc']:.3f}")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0].set_title(f"{TARGET_ORDER_KR} ({TARGET_ORDER_EN}) 공간 CV ROC", fontsize=9.5, fontweight="bold")
axes[0].set_xlabel("위양성률 (FPR)")
axes[0].set_ylabel("진양성률 (TPR)")
axes[0].legend()

for i, (vcode, vimp) in enumerate(top_vars[:3]):
    gx, gy = compute_response_curve(order_model, env_2020, vcode)
    axes[i + 1].plot(gx, gy, lw=2.2, color=GFO_COLORS["slate"])
    axes[i + 1].axhline(order_model["threshold"], color=GFO_COLORS["terracotta"], ls="--")
    axes[i + 1].set_title(f"{DRIVER_LABELS_KO[vcode]} ({vimp:.1f}%)", fontsize=9.0, fontweight="bold")
    axes[i + 1].set_ylim(0, 1)
plt.tight_layout()
plt.show()

pd.DataFrame([(k, DRIVER_LABELS_KO[k], round(v, 2)) for k, v in top_vars[:6]], columns=["변수_code", "환경_예측변수", "상대_중요도_pct"])

- `벌목(Hymenoptera)` 통합 모형: 50 km 공간 `block` 교차검증 `AUC = 0.621`, `TSS = 0.195`(`임계치 = 0.441`)
- 상위 중요 변수: 토지피복 다양성(`shannon`, `16.4%`), 산림–개방지 경계밀도(`edge_FO`, `13.8%`), 일사량(`solarradiation`, `10.0%`), 산림 비율(`frac_2`, `9.1%`) 순

## Step 2-B. 여름철 고온 한계를 반영한 2050년 분포 변화와 산지 방향 후퇴 (`Slides 19–26`)
- **미래 투영 및 기온 한계 제약**: 2050년 환경(`Biodiversity-SSP126`·`BAU-SSP585`) 투영 후 미관측 고온 과대예측 방지를 위해 관찰 기후 한계(`Thermal_Limits.json`; Araújo et al., 2013) 초과 격자를 부적합(`0`) 처리
- **임계치 기준**: 국내 지위 절단과 기후장 기준연도 편차 보정을 위해 하계 상한은 $T_{\max} = \max(P_{99}^{\text{KR}}, P_{99}^{\text{GBIF}})$(국내 KMA `1 km` vs. 전 지구 GBIF WorldClim `10'`), 동계 하한은 보수적 $P_{01}$ 적용(생리적 치사온도가 아닌 관찰 분포 기후 한계)
- **단일 사례**: **`FG03 뒤영벌속`** 2050년 `BAU-SSP585` 예측에서 하계 상한(`Bio5 >= 33.41 °C`) 반영 전·후 비교 및 4유형(부적합·소실·유지·신규) 지도화

In [ ]:
# [실습 2-B] FG03 뒤영벌속 2050년(BAU-SSP585) 투영: 여름 고온 한계 반영 전·후 비교 및 4유형 지도
env_bio126 = load_env_scenario("env_2050_Biodiversity_SSP126")
env_bau585 = load_env_scenario("env_2050_BAU_SSP585")

tau = bombus_model["threshold"]  # [실습 2-A2] 최종 모형 임계치 재사용
t_max, t_min = bombus_tl["hot_limit_C"], bombus_tl["cold_limit_C"]

# 1) 기온 한계 미반영(unmasked) vs. 반영(masked) 적합 면적 비교
hsi_2020_unmasked = bombus_model["hsi_2020_raw"]
bin_2020_unmasked = hsi_2020_unmasked >= tau
hsi_2020_masked = np.where((env_2020["bio5"] >= t_max) | (env_2020["bio6"] <= t_min), 0.0, hsi_2020_unmasked)
bin_2020_masked = hsi_2020_masked >= tau

X_2050_feat, _ = build_maxent_features(env_bau585["X_raw"], scaler=bombus_model["scaler"])
hsi_2050_unmasked = predict_suitability(bombus_model["params"], X_2050_feat, env_bau585["water_frac"])
exceed_thermal_585 = (env_bau585["bio5"] >= t_max) | (env_bau585["bio6"] <= t_min)  # 기온 한계 초과 격자
hsi_2050_masked = np.where(exceed_thermal_585, 0.0, hsi_2050_unmasked)

bin_2050_unmasked = hsi_2050_unmasked >= tau
bin_2050_masked = hsi_2050_masked >= tau

chg_unmasked = (bin_2050_unmasked.sum() - bin_2020_unmasked.sum()) / bin_2020_unmasked.sum() * 100.0
chg_masked = (bin_2050_masked.sum() - bin_2020_masked.sum()) / bin_2020_masked.sum() * 100.0
print(f"기온 한계 미반영: {bin_2020_unmasked.sum():,} -> {bin_2050_unmasked.sum():,} km² ({chg_unmasked:+.1f}%)")
print(f"하계 상한({t_max} °C) 반영: {bin_2020_masked.sum():,} -> {bin_2050_masked.sum():,} km² ({chg_masked:+.1f}%) | 고온 초과 격자: {exceed_thermal_585.sum():,}개")

# 2) 서식지 변화 4유형: 0=부적합, 1=소실, 2=유지, 3=신규
state_unmasked = np.zeros(len(hsi_2020_masked), dtype=np.int8)
state_unmasked[bin_2020_unmasked & (~bin_2050_unmasked)] = 1
state_unmasked[bin_2020_unmasked & bin_2050_unmasked] = 2
state_unmasked[(~bin_2020_unmasked) & bin_2050_unmasked] = 3

state_masked = np.zeros(len(hsi_2020_masked), dtype=np.int8)
state_masked[bin_2020_masked & (~bin_2050_masked)] = 1
state_masked[bin_2020_masked & bin_2050_masked] = 2
state_masked[(~bin_2020_masked) & bin_2050_masked] = 3

fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.0))
plot_4state_map(axes[0], state_unmasked, mask_2d, f"1. 고온 한계 미반영 ({chg_unmasked:+.1f}%)")
axes[1].imshow(vec_to_2d(exceed_thermal_585.astype(float), mask_2d), cmap="Reds", vmin=0, vmax=1)
axes[1].set_title(f"2. 2050 여름 고온 한계 초과 (Bio5 >= {t_max} °C)", fontsize=10.2, fontweight="bold")
axes[1].axis("off")
plot_4state_map(axes[2], state_masked, mask_2d, f"3. 고온 한계 반영 ({chg_masked:+.1f}%)")
plt.tight_layout()
plt.show()

- 2050년 `BAU-SSP585` 내륙 저지대 `39,589`개 격자(`46.8%`)가 여름 최고기온 상한(`33.41 °C`) 초과 → `FG03 뒤영벌속` 적합지 변화율이 미반영 `-37.2%`에서 반영 시 **`-51.2%`(`43,201 -> 21,089 km²`)**로 확대

In [ ]:
# [전체 비교 2-B] 12개 기능군 2050년 면적 변화율·평균 고도 이동·북상 거리 비교 (SSP1-2.6 vs. SSP5-8.5)
all_sub_proj_585 = {}  # Step 3 중첩 분석 재사용
summary_12 = []

for scode, skr, sen, oen in SUBGROUP_LIST:
    sids = pa_subgroups.loc[pa_subgroups["subgroup_kr"] == skr, "grid_id"].to_numpy()
    tl_s = thermal_limits["subgroups"][skr]
    m_s = train_and_validate_sdm(sids, env_2020, bg_grid, beta_reg=1.5, use_hinge=True)
    p126 = project_to_scenario(m_s, env_2020, env_bio126, bg_grid, tl_s["hot_limit_C"], tl_s["cold_limit_C"])
    p585 = project_to_scenario(m_s, env_2020, env_bau585, bg_grid, tl_s["hot_limit_C"], tl_s["cold_limit_C"])
    all_sub_proj_585[skr] = p585
    summary_12.append({
        "기능군_code": scode,
        "상위_목": oen,
        "기능군_국문명": skr,
        "공간CV_AUC": m_s["cv_metrics"]["auc"],
        "하계상한_C": tl_s["hot_limit_C"],
        "적합면적_2020_km2": p585["area_2020_km2"],
        "SSP126_변화율_pct": p126["pct_change"],
        "SSP585_변화율_pct": p585["pct_change"],
        "고도이동_585_m": p585["elev_shift_m"],  # 적합 격자 평균 해발고도 변화(주로 저지대 소실 기인)
        "북상이동_585_km": p585["north_shift_km"],
    })

pd.DataFrame(summary_12)

- **저온 적응군 급감(`BAU-SSP585`)**: `FG12 하늘소과 -55.7%`, `FG03 뒤영벌속 -51.2%`, `FG09 호랑나비과 -50.9%`, `FG02 재니등에과 -50.7%`, `FG06 단독성 야생벌류 -48.3%`, `FG07 온대 노린재류 -46.2%` (평균 고도 `+67.6 ~ +139.1 m` 상승)
- **아열대 해안군 확장**: `FG08 큰별노린재과`는 고도 변화(`-1.2 m`) 없이 해안·저지대 북상(`+24.3 km`)과 함께 **`+87.9%`** 확대

## Step 3. 식물–화분매개자 공존 서식지 변화와 공간 불일치 (`Slides 27–31`)
- **배경**: 단일 종 비생물적 SDM만으로는 먹이·기주식물과의 공간 불일치 포착 불가(Peng et al., 2025)
- **지표**: 서식 적합지 중첩도(**Schoener's $D$**; Warren et al., 2008) 및 2020년 공존 격자의 2050년 변화 4유형(Schweiger et al., 2008; Schleuning et al., 2016; Briggs & Anderson, 2026)

In [ ]:
# [실습 3] FG03 뒤영벌속 x 봄철·목본 핵심 밀원군: Schoener's D 및 공존 변화 4유형 산출
plant_focal_kr = "봄철·목본 핵심 밀원군"
plant_ids = pa_plants.loc[pa_plants["plant_group_kr"] == plant_focal_kr, "grid_id"].to_numpy()
plant_tl = thermal_limits["plants"][plant_focal_kr]

plant_model = train_and_validate_sdm(plant_ids, env_2020, bg_grid, beta_reg=1.5, use_hinge=True)
spring_plant_585 = project_to_scenario(plant_model, env_2020, env_bau585, bg_grid, plant_tl["hot_limit_C"], plant_tl["cold_limit_C"])
bombus_585 = all_sub_proj_585["뒤영벌속"]

# 1) 지리적 적합도 중첩도 Schoener's D = 1 - 0.5 * sum(|p_poll - p_plant|)
p_poll_20 = bombus_585["hsi_2020"] / bombus_585["hsi_2020"].sum()
p_plnt_20 = spring_plant_585["hsi_2020"] / spring_plant_585["hsi_2020"].sum()
d_2020 = 1.0 - 0.5 * np.sum(np.abs(p_poll_20 - p_plnt_20))

p_poll_50 = bombus_585["hsi_2050"] / bombus_585["hsi_2050"].sum()
p_plnt_50 = spring_plant_585["hsi_2050"] / spring_plant_585["hsi_2050"].sum()
d_2050 = 1.0 - 0.5 * np.sum(np.abs(p_poll_50 - p_plnt_50))

# 2) 2020년 공존 격자(co_2020)의 2050년 변화 4유형 분류
v20, v50 = bombus_585["bin_2020"], bombus_585["bin_2050"]
p20, p50 = spring_plant_585["bin_2020"], spring_plant_585["bin_2050"]
co_2020, co_2050 = v20 & p20, v50 & p50

mismatch_1d = np.zeros(len(v20), dtype=np.int8)  # 0: 2020년 비공존
mismatch_1d[co_2020 & v50 & p50] = 1        # 1: 공존 유지
mismatch_1d[co_2020 & v50 & (~p50)] = 2     # 2: 식물 소실(먹이 결핍)
mismatch_1d[co_2020 & (~v50) & p50] = 3     # 3: 화분매개자 소실(수분 공백)
mismatch_1d[co_2020 & (~v50) & (~p50)] = 4  # 4: 동시 소실

n_co20 = int(co_2020.sum())
co_chg = (int(co_2050.sum()) - n_co20) / n_co20 * 100.0
print(f"[FG03 뒤영벌속 x 봄철·목본 밀원군] Schoener's D: {d_2020:.2f} -> {d_2050:.2f} | 공존 면적 변화율: {co_chg:+.1f}%")
print(f"2020년 공존 {n_co20:,}개 격자 변화 -> 1.공존유지: {(mismatch_1d == 1).sum() / n_co20 * 100:.1f}%, 2.식물소실: {(mismatch_1d == 2).sum() / n_co20 * 100:.1f}%, 3.매개자소실: {(mismatch_1d == 3).sum() / n_co20 * 100:.1f}%, 4.동시소실: {(mismatch_1d == 4).sum() / n_co20 * 100:.1f}%")

cmap_mm = ListedColormap([GFO_COLORS["stone"], GFO_COLORS["pine"], GFO_COLORS["ochre"], GFO_COLORS["slate"], GFO_COLORS["terracotta"]])
norm_mm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], cmap_mm.N)

fig, ax = plt.subplots(figsize=(5.4, 5.2))
im = ax.imshow(vec_to_2d(mismatch_1d, mask_2d), cmap=cmap_mm, norm=norm_mm)
cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2, 3, 4], fraction=0.046, pad=0.03)
cbar.ax.set_yticklabels(["0: 2020년 비공존", "1: 공존 유지", "2: 식물 소실", "3: 화분매개자 소실", "4: 동시 소실"])
ax.set_title("FG03 뒤영벌속 × 봄철·목본 밀원군 (2050 BAU-SSP585)", fontsize=10.2, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.show()

- `봄철·목본 핵심 밀원군`의 저지대 잔존과 `FG03 뒤영벌속`의 산지 후퇴(`Bio5 >= 33.41 °C`) 불일치 → `Schoener's D` `0.84 -> 0.64` 하락, 2020년 공존지의 **`23.1%`**가 **수분 공백(`3: 화분매개자 소실`)** 전환(2050년 밀원식물 적합지 전체 대비로는 `54.0%`에서 뒤영벌 부재)

In [ ]:
# [전체 비교 3] 5개 식물 기능군 SDM 적합 및 8개 핵심 상호작용 쌍 공존 변화 비교
plant_proj_585 = {plant_focal_kr: spring_plant_585}
for pkr, pmeta in thermal_limits["plants"].items():
    if pkr in plant_proj_585:
        continue
    pids = pa_plants.loc[pa_plants["plant_group_kr"] == pkr, "grid_id"].to_numpy()
    pm = train_and_validate_sdm(pids, env_2020, bg_grid, beta_reg=1.5, use_hinge=True)
    plant_proj_585[pkr] = project_to_scenario(pm, env_2020, env_bau585, bg_grid, pmeta["hot_limit_C"], pmeta["cold_limit_C"])

pairs = [
    ("FG03 뒤영벌속 × 봄철·목본 밀원군", all_sub_proj_585["뒤영벌속"], plant_proj_585["봄철·목본 핵심 밀원군"]),
    ("FG06 단독성 야생벌류 × 봄철·목본 밀원군", all_sub_proj_585["단독성 야생벌류"], plant_proj_585["봄철·목본 핵심 밀원군"]),
    ("FG01 꽃등에과 × 초지 국화과·산형과 밀원군", all_sub_proj_585["꽃등에과"], plant_proj_585["초지 국화과·산형과 밀원군"]),
    ("FG02 재니등에과 × 초지·습초지 식물군", all_sub_proj_585["재니등에과"], plant_proj_585["초지·습초지 식물군"]),
    ("FG09 호랑나비과 × 산림성 식물군", all_sub_proj_585["호랑나비과"], plant_proj_585["산림성 식물군"]),
    ("FG10 네발나비과 × 콩과·제비꽃 기주·밀원군", all_sub_proj_585["네발나비과"], plant_proj_585["콩과·제비꽃 기주·밀원군"]),
    ("FG07 온대 노린재류 × 초지·습초지 식물군", all_sub_proj_585["온대 노린재류"], plant_proj_585["초지·습초지 식물군"]),
    ("FG12 하늘소과 × 산림성 식물군", all_sub_proj_585["하늘소과"], plant_proj_585["산림성 식물군"]),
]

ov_rows = []
for label, vp, pp in pairs:
    ov = analyze_bipartite_overlap(vp, pp)
    ov_rows.append({
        "상호작용_쌍": label,
        "공존면적_변화율_2050_pct": ov["cooccur_change_pct"],
        "Schoener_D_20to50": f"{ov['schoeners_d_2020']:.2f} -> {ov['schoeners_d_2050']:.2f}",
        "공존유지_pct": ov["state_1_persist_pct"],
        "식물소실_먹이결핍_pct": ov["state_2_plant_lost_pct"],
        "화분매개자소실_수분공백_pct": ov["state_3_poll_lost_pct"],
        "동시소실_pct": ov["state_4_both_lost_pct"],
    })

pd.DataFrame(ov_rows)

- **먹이 결핍**: `FG01 꽃등에과`(`-30.9%`)와 `초지 국화과·산형과 밀원군`(`-23.9%`) 감소 중첩으로 공존지 **`-34.2%`** 축소(`식물 소실` `17.9%`)
- **수분 공백**: 산지 후퇴 폭이 큰 `FG03 뒤영벌속`(`23.1%`), `FG09 호랑나비과`(`22.9%`), `FG12 하늘소과`(`21.0%`) 쌍에서 2020년 공존지의 20% 이상 수분 공백 전환(`Slide 30`)

In [ ]:
# [전체 비교 3-B] 274종 식물 x 139종 화분매개자(SSDM) 군집 수준 공존 피난처 요약 (Slide 31)
comm_regimes = pd.read_csv(os.path.join(WORKSHOP_ROOT, "outputs", "step3_community_regimes_summary.csv"))
comm_regimes.rename(columns={
    "Scope": "분석_범위_및_scenario",
    "Valid_Cells_S_ge_5": "유효격자수_S5이상",
    "Low_Baseline_lt5_Cells": "저밀도제외_격자수",
    "Refugia_pct": "공존피난처_유지_pct",
    "Pollinator_Deficit_pct": "화분매개자결핍_pct",
    "Floral_Host_Deficit_pct": "밀원기주식물결핍_pct",
    "Dual_Loss_pct": "양측동반손실_pct",
})

- **군집 공존 피난처(`Slide 31`, 기저 $S_{2020} \ge 5$ 유효 격자 `76,575`개·양측 $\Delta S \ge -20\%$ 유지 기준)**: **`Biodiversity-SSP126`**(`29.9%`)이 **`BAU-SSP585`**(`15.2%`) 대비 약 2배 유지; `BAU-SSP585` 해발 `200–500 m` 산록부의 **`51.1%`**가 밀원·기주 결핍 전환

## Step 4. 군집 종조성 변화: 적층 종분포모형(SSDM)과 일반화 비유사도 모형(GDM) (`Slides 32–41`)
- **`SSDM`**(Dubuis et al., 2011; Calabrese et al., 2014): 개별 종 적합도 적층으로 종풍부도 변화($\Delta S$)와 지점별 종 교체·종 감소 산출
- **`GDM`**(Ferrier et al., 2007; Mokany et al., 2022): 지점 쌍 비유사도($d_{ij} = 1 - e^{-\eta_{ij}}$)를 3-매듭 단조 `I-spline`으로 적합해 환경 구배별 비선형 급변 구간 및 누적 변화량($H_k = \sum_{m=1}^3 a_{km}$) 포착
- **Baselga (2010) 분할**: 전체 Sørensen 비유사도 $\beta_{\text{sor}} = (b + c)/(2a + b + c)$ = **종 교체** $\beta_{\text{sim}} = \min(b, c)/[a + \min(b, c)]$(1:1 종 치환) + **종 감소** $\beta_{\text{sne}} = \beta_{\text{sor}} - \beta_{\text{sim}}$(대체 유입 없는 부분집합화)
- **해석 한계**: 공간–시간 치환 및 무제한 산포 가정에 따른 잠재–실현 군집 괴리 가능(Fitzpatrick & Keller, 2015); 기후 외삽 진단(**MESS** / **ExDet**) 병행 필요

![개념 비교: SSDM(개별 종 적층) vs. GDM(지점 쌍 비유사도 모형화)](figures/comics/concept_sdm_vs_gdm.png)

![종조성 변화를 보는 두 관점: 환경 구배 I-spline과 Baselga 집합 분할](figures/comics/concept_turnover_explained.png)

### Step 4 단일 사례: 두 조사 지점 Baselga 분할 및 기온 `I-spline` 변화곡선
- 관측 육상식물 행렬(`GDM_Community_Benchmark.npz`)의 두 격자(`Site 0`, `Site 15`) 간 공유종 `a`·고유종 `b`, `c`로 $\beta_{\text{sor}} = \beta_{\text{sim}} + \beta_{\text{sne}}$ 직접 확인

In [ ]:
# [실습 4-A] 두 조사 지점(Site 0 vs. Site 15) Sørensen 비유사도 및 Baselga 분할 계산
bm = np.load(os.path.join(DATA_DIR, "GDM_Community_Benchmark.npz"))
plant_s5_ids, plant_s5_mat = bm["plant_obs_s5_ids"], bm["plant_obs_s5_mat"]  # 5종 이상 기록 식물 격자(274종)

site_i, site_j = plant_s5_mat[0] > 0, plant_s5_mat[15] > 0
a = int(np.sum(site_i & site_j))        # 공유 종수
b = int(np.sum(site_i & (~site_j)))     # Site 0 고유 종수
c = int(np.sum((~site_i) & site_j))     # Site 15 고유 종수

beta_sor = (b + c) / (2 * a + b + c)
beta_sim = min(b, c) / (a + min(b, c))  # 순수 종 교체(Simpson)
beta_sne = beta_sor - beta_sim          # 종수 차이에 따른 종 감소(중첩성)

print(f"Site 0 (S = {int(site_i.sum())}) vs. Site 15 (S = {int(site_j.sum())}): a = {a}, b = {b}, c = {c}")
print(f"beta_sor = {beta_sor:.3f}  ==  종 교체(beta_sim) {beta_sim:.3f} + 종 감소(beta_sne) {beta_sne:.3f}")

- 두 지점(`Site 0` `S = 9`, `Site 15` `S = 47`; `a = 3, b = 6, c = 44`): 전체 비유사도($\beta_{\text{sor}} = 0.893$) = 종 교체($\beta_{\text{sim}} = 0.667$) + 종 감소($\beta_{\text{sne}} = 0.226$) 일치 확인

In [ ]:
# [실습 4-B] 139종 화분매개자 SSDM 군집 GDM 적합 및 연평균기온(bioclim1) I-spline 곡선 산출
X_2020 = env_2020["X_raw"]
coords_km = bg_grid[["x_5186", "y_5186"]].to_numpy(dtype=np.float64) / 1000.0
ssdm_ids, poll_ssdm_mat = bm["ssdm_sample_ids"], bm["poll_ssdm_sample_mat"].astype(np.float32)

gdm_ssdm = fit_gdm_ispline(poll_ssdm_mat, X_2020[ssdm_ids], coords_km[ssdm_ids], DEFAULT_DRIVERS, metric="braycurtis", max_pairs=25000)

vcode = "bioclim1"
knots_b1 = gdm_ssdm["knots"][vcode]          # 최소·중앙·최대 3개 매듭 온도
coefs_b1 = gdm_ssdm["var_coefs_3"][vcode]    # NNLS 비음수 기저 가중치(a1, a2, a3 >= 0)
height_b1 = float(np.sum(coefs_b1))          # I-spline 최대 높이 H_k = a1 + a2 + a3

x_grid = np.linspace(knots_b1[0], knots_b1[2], 200)
I_curve = eval_ispline_3(x_grid, knots_b1) @ coefs_b1  # 부분 생태적 거리 곡선 I_k(x)

print(f"[GDM bioclim1] 3-매듭: ({knots_b1[0]:.1f}, {knots_b1[1]:.1f}, {knots_b1[2]:.1f}) °C")
print(f"비음수 가중치 (a1, a2, a3): {np.round(coefs_b1, 3)} -> 최대 높이 H_k = {height_b1:.3f} (환경 변화 비중 {gdm_ssdm['var_env_imp'][vcode]:.1f}%)")

fig, ax = plt.subplots(figsize=(5.8, 3.4))
ax.plot(x_grid, I_curve, lw=2.5, color=GFO_COLORS["terracotta"], label=f"I(bioclim1), H = {height_b1:.2f}")
for kn in knots_b1:
    ax.axvline(kn, color="gray", ls=":", lw=0.9)
ax.set_xlabel("연평균기온 (bioclim1, °C)")
ax.set_ylabel("누적 종조성 변화 I(x)")
ax.set_title("연평균기온에 따른 종조성 변화곡선 (GDM I-spline)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

- `bioclim1` 3개 분위수 매듭 `(6.1, 12.6, 16.5) °C` 기준 최대 높이 $H_k = 0.608$(환경 비중 `10.7%`), 저온~중온 전이 구간에서 부분 생태적 거리 $I(x)$ 급증

### Step 4 전체 비교: 조사 완전성별 GDM 설명력 및 2050년 전국 군집 변화 지도
- **4가지 군집 자료 체계** 간 GDM 설명력 비교 및 2050년(`BAU-SSP585`) 전국 **GDM 종조성 변화**·**SSDM Baselga 분할** 지도 비교

In [ ]:
# [전체 비교 4-A] 조사 완전성이 다른 4가지 군집 행렬의 GDM 설명력 비교
poll_all_ids, poll_all_mat = bm["poll_obs_all_ids"], bm["poll_obs_all_mat"]
mask_s10 = np.sum(poll_all_mat > 0, axis=1) >= 10  # 격자당 10종 이상 기록된 고밀도 조사 격자

gdm_raw = fit_gdm_ispline(poll_all_mat, X_2020[poll_all_ids], coords_km[poll_all_ids], DEFAULT_DRIVERS, metric="sorensen", max_pairs=25000)
gdm_s10 = fit_gdm_ispline(poll_all_mat[mask_s10], X_2020[poll_all_ids[mask_s10]], coords_km[poll_all_ids[mask_s10]], DEFAULT_DRIVERS, metric="sorensen", use_weights=True, max_pairs=25000)
gdm_plant = fit_gdm_ispline(plant_s5_mat, X_2020[plant_s5_ids], coords_km[plant_s5_ids], DEFAULT_DRIVERS, metric="sorensen", use_weights=True, max_pairs=25000)

pd.DataFrame([
    {"모형_체계": "1. 원시 곤충 출현 행렬 (1 km PO, S>=1)", "조사_격자수": gdm_raw["n_sites"], "공유종없는_쌍_비율_d1_pct": gdm_raw["pct_saturated_d1"], "모형_설명력_pct": gdm_raw["pct_dev_explained"], "상관계수_r": gdm_raw["pearson_r"]},
    {"모형_체계": "2. 선별 곤충 출현 행렬 (1 km PO, S>=10)", "조사_격자수": gdm_s10["n_sites"], "공유종없는_쌍_비율_d1_pct": gdm_s10["pct_saturated_d1"], "모형_설명력_pct": gdm_s10["pct_dev_explained"], "상관계수_r": gdm_s10["pearson_r"]},
    {"모형_체계": "3. 관측 육상식물 출현 행렬 (1 km PO, S>=5)", "조사_격자수": gdm_plant["n_sites"], "공유종없는_쌍_비율_d1_pct": gdm_plant["pct_saturated_d1"], "모형_설명력_pct": gdm_plant["pct_dev_explained"], "상관계수_r": gdm_plant["pearson_r"]},
    {"모형_체계": "4. 화분매개자 SSDM 적층 군집 (139종)", "조사_격자수": gdm_ssdm["n_sites"], "공유종없는_쌍_비율_d1_pct": gdm_ssdm["pct_saturated_d1"], "모형_설명력_pct": gdm_ssdm["pct_dev_explained"], "상관계수_r": gdm_ssdm["pearson_r"]},
])

- 단회 방문 원시 곤충 출현 행렬(`S >= 1`)은 지점 쌍 `45.5%`가 공유종 부재($d_{ij} = 1$)로 포화되어 GDM 설명력 `0.81%`에 그치는 반면, 육상식물 관측 행렬(`37.04%`; 영국 반복 조사 기반 GDM `31–56%`[Cooke et al., 2026]에 대응)과 139종 SSDM 적층 군집(`73.70%`, $r = 0.871$)은 연속적 종조성 변화를 안정 포착

In [ ]:
# [전체 비교 4-B] 2050년(BAU-SSP585) 전국 GDM 종조성 변화 및 SSDM Baselga 분할 지도 비교
comm_dir = os.path.join(DATA_DIR, "community_richness")
def read_comm_tif(fn):
    with rasterio.open(os.path.join(comm_dir, fn)) as src:
        return src.read(1)[mask_2d].astype(np.float64)

gdm_585 = read_comm_tif("GDM_TemporalTurnover_Poll_SSDM_BAU-SSP585.tif")
sor_585 = read_comm_tif("SSDM_Baselga_BetaSor_Poll_BAU-SSP585.tif")
sim_585 = read_comm_tif("SSDM_Baselga_BetaSim_Poll_BAU-SSP585.tif")
sne_585 = read_comm_tif("SSDM_Baselga_BetaSne_Poll_BAU-SSP585.tif")

elev_1d = bg_grid["elevation_m"].to_numpy()
low_mask = (elev_1d < 200) & (sor_585 > 0)
high_mask = (elev_1d >= 800) & (sor_585 > 0)
print(f"저지대 (<200 m): 평균 beta_sor = {sor_585[low_mask].mean():.3f} (종 감소 beta_sne = {sne_585[low_mask].mean():.3f}, 종 교체 beta_sim = {sim_585[low_mask].mean():.3f})")
print(f"고지대 (>=800 m): 평균 beta_sor = {sor_585[high_mask].mean():.3f} (종 감소 beta_sne = {sne_585[high_mask].mean():.3f}, 종 교체 beta_sim = {sim_585[high_mask].mean():.3f})")

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))
map_specs = [
    (axes[0], gdm_585, "1. GDM 종조성 변화 (2020-2050)", "YlOrRd", 0.45),
    (axes[1], sim_585, "2. SSDM 종 교체 (beta_sim)", "PuBuGn", 0.35),
    (axes[2], sne_585, "3. SSDM 종 감소 (beta_sne)", "OrRd", 0.80),
]
for ax, arr, ttl, cm, vm in map_specs:
    im = ax.imshow(vec_to_2d(arr, mask_2d), cmap=cm, vmin=0, vmax=vm)
    ax.set_title(f"{ttl}\n(전국 평균 {np.mean(arr):.2f})", fontsize=10.0, fontweight="bold")
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
plt.tight_layout()
plt.show()

- **저지대(`<200 m`, 유효 격자)**: 평균 비유사도($\beta_{\text{sor}} = 0.753$)의 `85.1%`가 대체 종 유입 없는 **종 감소**($\beta_{\text{sne}} = 0.641$ vs. $\beta_{\text{sim}} = 0.112$)
- **고지대(`>=800 m`)**: 평균 비유사도($\beta_{\text{sor}} = 0.280$) 중 상향 유입에 따른 **종 교체**($\beta_{\text{sim}} = 0.167$, `59.7%` vs. $\beta_{\text{sne}} = 0.113$) 우세

### Key Findings (`Slides 42–43`)
- **기능군별 차별적 취약성**: 벌목 통합 SDM(`-23.9%`, `outputs/step2_orders_summary.csv`)에 가려진 산지 저온성 `FG03 뒤영벌속`(`-51.2%`)·`FG06 단독성 야생벌류`(`-48.3%`) 급감 확인(광온성 `FG04 꿀벌속` `-23.2%`); 평균 고도 상승은 대부분 저지대 하한선 소실에 기인하며, 남방 해안성 `FG08 큰별노린재과`는 해안을 따라 수평 북상(`+24.3 km`, `+87.9%`)
- **100 m 경관 이질성의 중요성**: 100 m 토지피복·경관구조 6종(`frac_0..3`, `edge_FO`, `shannon`)이 SDM 가중치 합의 `38.6%`·SSDM-GDM 환경 `spline` 높이 합의 `36.4%` 차지(`outputs/step4_gdm_variable_importance.csv`) → 산림 영소처와 개방지 밀원을 잇는 임연부 경계밀도(`edge_FO`)의 핵심 역할 확인(Westrich, 1996; Tscharntke et al., 2005)
- **기온 한계($P_{99}$) 결합 보정**: 전 지구 GBIF(`10'`)와 국내 NES(`1 km`) 결합 상한으로 지위 절단 및 기후장 편차를 보정(`FG03 뒤영벌속` 2050 `BAU-SSP585`: 미반영 `-37.2%` → 반영 `-51.2%`)
- **식물–화분매개자 불일치와 보전 효과**: 초지 밀원 감소 중첩(`FG01` 공존지 `-34.2%`) 및 목본 잔존–곤충 산지 후퇴 불일치(`FG03`·`FG09` 공존지의 `22.9–23.1%` 수분 공백) 발생; `Biodiversity-SSP126`은 전국 공존 피난처를 `29.9%` 유지해 `BAU-SSP585`(`15.2%`) 대비 약 2배 보전
- **저지대 생물상 빈약화 vs. 고지대 종 교체**: 해양으로 격리된 반도 저지대(`<200 m`)는 남방계 대체 종 유입 제한으로 **종 감소**(`~85%`, $\beta_{\text{sne}}$) 지배, 고지대(`>=800 m`)는 상향 유입 **종 교체**(`59.7%`, $\beta_{\text{sim}}$) 우세

### Further work
- **단일 기능군·정규화 민감도 실습**: `[실습 1-A]`~`[실습 2-B]` 대상을 `FG04`(`꿀벌속`)·`FG08`(`큰별노린재과`)로 바꾸거나 `beta_reg`(`0.5` vs. `3.0`) 조정 후 반응곡선·2050년 분포 비교
- **`Biodiversity-SSP126` 상호작용 완화 검정**: `env_bio126`을 `[실습 3]`·`[전체 비교 3]`에 대입해 `BAU-SSP585` 대비 공존 유지율 및 수분 공백 감소 폭 비교